ECIF feature generation 

In [1]:

import numpy as np
import pandas as pd
from os import listdir
from rdkit import Chem
from scipy.spatial.distance import cdist
from itertools import product
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

In [2]:

# Possible predefined protein atoms
ECIF_ProteinAtoms = ['C;4;1;3;0;0', 'C;4;2;1;1;1', 'C;4;2;2;0;0', 'C;4;2;2;0;1',
                     'C;4;3;0;0;0', 'C;4;3;0;1;1', 'C;4;3;1;0;0', 'C;4;3;1;0;1',
                     'C;5;3;0;0;0', 'C;6;3;0;0;0', 'N;3;1;2;0;0', 'N;3;2;0;1;1',
                     'N;3;2;1;0;0', 'N;3;2;1;1;1', 'N;3;3;0;0;1', 'N;4;1;2;0;0',
                     'N;4;1;3;0;0', 'N;4;2;1;0;0', 'O;2;1;0;0;0', 'O;2;1;1;0;0',
                     'S;2;1;1;0;0', 'S;2;2;0;0;0']

# Possible ligand atoms according to the PDBbind 2016 "refined set"
ECIF_LigandAtoms = ['Br;1;1;0;0;0', 'C;3;3;0;1;1', 'C;4;1;1;0;0', 'C;4;1;2;0;0',
                     'C;4;1;3;0;0', 'C;4;2;0;0;0', 'C;4;2;1;0;0', 'C;4;2;1;0;1',
                     'C;4;2;1;1;1', 'C;4;2;2;0;0', 'C;4;2;2;0;1', 'C;4;3;0;0;0',
                     'C;4;3;0;0;1', 'C;4;3;0;1;1', 'C;4;3;1;0;0', 'C;4;3;1;0;1',
                     'C;4;4;0;0;0', 'C;4;4;0;0;1', 'C;5;3;0;0;0', 'C;5;3;0;1;1',
                     'C;6;3;0;0;0', 'Cl;1;1;0;0;0', 'F;1;1;0;0;0', 'I;1;1;0;0;0',
                     'N;3;1;0;0;0', 'N;3;1;1;0;0', 'N;3;1;2;0;0', 'N;3;2;0;0;0',
                     'N;3;2;0;0;1', 'N;3;2;0;1;1', 'N;3;2;1;0;0', 'N;3;2;1;0;1',
                     'N;3;2;1;1;1', 'N;3;3;0;0;0', 'N;3;3;0;0;1', 'N;3;3;0;1;1',
                     'N;4;1;2;0;0', 'N;4;1;3;0;0', 'N;4;2;1;0;0', 'N;4;2;2;0;0',
                     'N;4;2;2;0;1', 'N;4;3;0;0;0', 'N;4;3;0;0;1', 'N;4;3;1;0;0',
                     'N;4;3;1;0;1', 'N;4;4;0;0;0', 'N;4;4;0;0;1', 'N;5;2;0;0;0',
                     'N;5;3;0;0;0', 'N;5;3;0;1;1', 'O;2;1;0;0;0', 'O;2;1;1;0;0',
                     'O;2;2;0;0;0', 'O;2;2;0;0;1', 'O;2;2;0;1;1', 'P;5;4;0;0;0',
                     'P;6;4;0;0;0', 'P;6;4;0;0;1', 'P;7;4;0;0;0', 'S;2;1;0;0;0',
                     'S;2;1;1;0;0', 'S;2;2;0;0;0', 'S;2;2;0;0;1', 'S;2;2;0;1;1',
                     'S;3;3;0;0;0', 'S;3;3;0;0;1', 'S;4;3;0;0;0', 'S;6;4;0;0;0',
                     'S;6;4;0;0;1', 'S;7;4;0;0;0']

PossibleECIF = [i[0]+"-"+i[1] for i in product(ECIF_ProteinAtoms, ECIF_LigandAtoms)]

In [3]:
ELEMENTS_ProteinAtoms = ["C","N","O", "S"]
ELEMENTS_LigandAtoms = ["Br", "C", "Cl", "F", "I", "N", "O", "P", "S"]
PossibleELEMENTS = [i[0]+"-"+i[1] for i in product(ELEMENTS_ProteinAtoms, ELEMENTS_LigandAtoms)]

In [4]:
LigandDescriptors = ['MaxEStateIndex', 'MinEStateIndex', 'MaxAbsEStateIndex', 'MinAbsEStateIndex',
                      'qed', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons',
                      'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BalabanJ',
                      'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n',
                      'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha', 'Kappa1',
                      'Kappa2', 'Kappa3', 'LabuteASA', 'PEOE_VSA14', 'SMR_VSA1', 'SMR_VSA10',
                      'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7',
                      'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12',
                      'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6',
                      'SlogP_VSA7', 'SlogP_VSA8', 'TPSA', 'EState_VSA1', 'EState_VSA10',
                      'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5',
                      'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'VSA_EState1',
                      'VSA_EState10', 'VSA_EState2', 'VSA_EState3', 'VSA_EState4', 'VSA_EState5',
                      'VSA_EState6', 'VSA_EState7', 'VSA_EState8', 'VSA_EState9', 'FractionCSP3',
                      'HeavyAtomCount', 'NHOHCount', 'NOCount', 'NumAliphaticCarbocycles',
                      'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAromaticCarbocycles',
                      'NumAromaticHeterocycles', 'NumAromaticRings', 'NumHAcceptors', 'NumHDonors',
                      'NumHeteroatoms', 'NumRotatableBonds', 'NumSaturatedCarbocycles',
                      'NumSaturatedHeterocycles', 'NumSaturatedRings', 'RingCount', 'MolLogP',
                      'MolMR', 'fr_Al_COO', 'fr_Al_OH', 'fr_Al_OH_noTert', 'fr_ArN', 'fr_Ar_N',
                      'fr_Ar_NH', 'fr_Ar_OH', 'fr_COO', 'fr_COO2', 'fr_C_O', 'fr_C_O_noCOO',
                      'fr_C_S', 'fr_HOCCN', 'fr_Imine', 'fr_NH0', 'fr_NH1', 'fr_NH2', 'fr_N_O',
                      'fr_Ndealkylation1', 'fr_Ndealkylation2', 'fr_Nhpyrrole', 'fr_SH', 'fr_aldehyde',
                      'fr_alkyl_carbamate', 'fr_alkyl_halide', 'fr_allylic_oxid', 'fr_amide',
                      'fr_amidine', 'fr_aniline', 'fr_aryl_methyl', 'fr_azo', 'fr_barbitur',
                      'fr_benzene', 'fr_bicyclic', 'fr_dihydropyridine', 'fr_epoxide', 'fr_ester',
                      'fr_ether', 'fr_furan', 'fr_guanido', 'fr_halogen', 'fr_hdrzine', 'fr_hdrzone',
                      'fr_imidazole', 'fr_imide', 'fr_isocyan', 'fr_isothiocyan', 'fr_ketone',
                      'fr_ketone_Topliss', 'fr_lactam', 'fr_lactone', 'fr_methoxy', 'fr_morpholine',
                      'fr_nitrile', 'fr_nitro', 'fr_nitro_arom', 'fr_nitroso', 'fr_oxazole',
                      'fr_oxime', 'fr_para_hydroxylation', 'fr_phenol', 'fr_phenol_noOrthoHbond',
                      'fr_piperdine', 'fr_piperzine', 'fr_priamide', 'fr_pyridine', 'fr_quatN',
                      'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene', 'fr_tetrazole',
                      'fr_thiazole', 'fr_thiocyan', 'fr_thiophene', 'fr_urea']

DescCalc = MolecularDescriptorCalculator(LigandDescriptors)

In [5]:
def GetAtomType(atom):
# This function takes an atom in a molecule and returns its type as defined for ECIF
    
    AtomType = [atom.GetSymbol(),
                str(atom.GetExplicitValence()),
                str(len([x.GetSymbol() for x in atom.GetNeighbors() if x.GetSymbol() != "H"])),
                str(len([x.GetSymbol() for x in atom.GetNeighbors() if x.GetSymbol() == "H"])),
                str(int(atom.GetIsAromatic())),
                str(int(atom.IsInRing())), 
               ]

    return(";".join(AtomType))

In [6]:
def LoadSDFasDF(SDF):
# This function takes an SDF for a ligand as input and returns it as a pandas DataFrame with its atom types labeled according to ECIF
    
    m = Chem.MolFromMolFile(SDF, sanitize=False)
    m.UpdatePropertyCache(strict=False)
    
    ECIF_atoms = []

    for atom in m.GetAtoms():
        if atom.GetSymbol() != "H": # Include only non-hydrogen atoms
            entry = [int(atom.GetIdx())]
            entry.append(GetAtomType(atom))
            pos = m.GetConformer().GetAtomPosition(atom.GetIdx())
            entry.append(float("{0:.4f}".format(pos.x)))
            entry.append(float("{0:.4f}".format(pos.y)))
            entry.append(float("{0:.4f}".format(pos.z)))
            ECIF_atoms.append(entry)

    df = pd.DataFrame(ECIF_atoms)
    df.columns = ["ATOM_INDEX", "ECIF_ATOM_TYPE","X","Y","Z"]
    if len(set(df["ECIF_ATOM_TYPE"]) - set(ECIF_LigandAtoms)) > 0:
        print("WARNING: Ligand contains unsupported atom types. Only supported atom-type pairs are counted.")
    return(df)


In [7]:
Atom_Keys=pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/PDB_Atom_Keys.csv", sep=",")
def LoadPDBasDF(PDB):
# This function takes a PDB for a protein as input and returns it as a pandas DataFrame with its atom types labeled according to ECIF

    ECIF_atoms = []
    
    f = open(PDB)
    for i in f:
        if i[:4] == "ATOM":
            # Include only non-hydrogen atoms
            if (len(i[12:16].replace(" ","")) < 4 and i[12:16].replace(" ","")[0] != "H") or (len(i[12:16].replace(" ","")) == 4 and i[12:16].replace(" ","")[1] != "H" and i[12:16].replace(" ","")[0] != "H"):
                ECIF_atoms.append([int(i[6:11]),
                         i[17:20]+"-"+i[12:16].replace(" ",""),
                         float(i[30:38]),
                         float(i[38:46]),
                         float(i[46:54])
                        ])
                
    f.close()
    
    df = pd.DataFrame(ECIF_atoms, columns=["ATOM_INDEX","PDB_ATOM","X","Y","Z"])
    df = df.merge(Atom_Keys, left_on='PDB_ATOM', right_on='PDB_ATOM')[["ATOM_INDEX", "ECIF_ATOM_TYPE", "X", "Y", "Z"]].sort_values(by="ATOM_INDEX").reset_index(drop=True)
    if list(df["ECIF_ATOM_TYPE"].isna()).count(True) > 0:
        print("WARNING: Protein contains unsupported atom types. Only supported atom-type pairs are counted.")
    return(df)

In [8]:
def GetPLPairs(PDB_protein, SDF_ligand, distance_cutoff=6.0):
# This function returns the protein-ligand atom-type pairs for a given distance cutoff
    
    # Load both structures as pandas DataFrames
    Target = LoadPDBasDF(PDB_protein)
    Ligand = LoadSDFasDF(SDF_ligand)
    
    # Take all atoms from the target within a cubic box around the ligand considering the "distance_cutoff criterion"
    for i in ["X","Y","Z"]:
        Target = Target[Target[i] < float(Ligand[i].max())+distance_cutoff]
        Target = Target[Target[i] > float(Ligand[i].min())-distance_cutoff]
    
    # Get all possible pairs
    Pairs = list(product(Target["ECIF_ATOM_TYPE"], Ligand["ECIF_ATOM_TYPE"]))
    Pairs = [x[0]+"-"+x[1] for x in Pairs]
    Pairs = pd.DataFrame(Pairs, columns=["ECIF_PAIR"])
    Distances = cdist(Target[["X","Y","Z"]], Ligand[["X","Y","Z"]], metric="euclidean")
    Distances = Distances.reshape(Distances.shape[0]*Distances.shape[1],1)
    Distances = pd.DataFrame(Distances, columns=["DISTANCE"])

    Pairs = pd.concat([Pairs,Distances], axis=1)
    Pairs = Pairs[Pairs["DISTANCE"] <= distance_cutoff].reset_index(drop=True)
    # Pairs from ELEMENTS could be easily obtained froms pairs from ECIF
    Pairs["ELEMENTS_PAIR"] = [x.split("-")[0].split(";")[0]+"-"+x.split("-")[1].split(";")[0] for x in Pairs["ECIF_PAIR"]]
    return Pairs

In [9]:
def GetECIF(PDB_protein, SDF_ligand, distance_cutoff=6.0):
    # Main function for the calculation of ECIF
    Pairs = GetPLPairs(PDB_protein, SDF_ligand, distance_cutoff=distance_cutoff)
    print(f"{prefix}: {len(Pairs)} pairs found")
    # Check if Pairs is empty
    if Pairs.empty:
        return None  # Nothing to calculate
    
    ECIF = [list(Pairs["ECIF_PAIR"]).count(x) for x in PossibleECIF]
    return ECIF


In [10]:
def GetELEMENTS(PDB_protein, SDF_ligand, distance_cutoff=6.0):
# Function for the calculation of ELEMENTS
    Pairs = GetPLPairs(PDB_protein, SDF_ligand, distance_cutoff=distance_cutoff)
    ELEMENTS = [list(Pairs["ELEMENTS_PAIR"]).count(x) for x in PossibleELEMENTS]
    return ELEMENTS

In [11]:
def GetRDKitDescriptors(SDF):
# Function for the calculation of ligand descriptors
    mol = Chem.MolFromMolFile(SDF, sanitize=False)
    mol.UpdatePropertyCache(strict=False)
    Chem.GetSymmSSSR(mol)
    return DescCalc.CalcDescriptors(mol)

Generation ECIF descriptors for training data

In [79]:
import os
import pandas as pd

protein_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/proteins_pdb_train"
ligand_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/ligand_train_rdkit"

ligand_files = [f for f in os.listdir(ligand_dir) if f.endswith(".sdf")]
results = []

for ligand_file in ligand_files:
    prefix = ligand_file.split("_")[0]  # e.g., "1a1b"
    protein_file = os.path.join(protein_dir, f"{prefix}_receptor.pdb")
    ligand_path = os.path.join(ligand_dir, ligand_file)

    if not os.path.exists(protein_file):
        print(f"⚠️ Protein file not found for {prefix}, skipping...")
        continue

    print(f"✅ Processing {prefix} ...")

    # === TRY-EXCEPT to skip any problematic complex ===
    try:
        ecif_vector = GetECIF(protein_file, ligand_path)
        if ecif_vector is None or len(ecif_vector) == 0:
            print(f"⚠️ Skipping {prefix} — no ECIF features generated.")
            continue
        # Add Complex ID as first column
        ecif_vector = [prefix] + ecif_vector
        results.append(ecif_vector)

    except Exception as e:
        print(f"❌ Skipping {prefix} due to error: {e}")
        continue

# Convert to DataFrame
if results:
    # First column = Complex_ID, rest = ECIF headers
    headers = ["Complex_ID"] + PossibleECIF
    df_ecif = pd.DataFrame(results, columns=headers)
    df_ecif.to_csv("ECIF_descriptors_train.csv", index=False)
    print(f"\n✅ ECIF descriptors saved to: ECIF_descriptors_train.csv")
else:
    print("❌ No ECIF features generated. Check your input files.")


✅ Processing 3r02 ...
3r02: 488 pairs found
✅ Processing 6c2t ...
6c2t: 674 pairs found
✅ Processing 5t31 ...
5t31: 442 pairs found
✅ Processing 4eop ...
4eop: 693 pairs found
✅ Processing 2ghg ...
2ghg: 735 pairs found
✅ Processing 4njd ...
4njd: 610 pairs found
✅ Processing 4acm ...
4acm: 819 pairs found
✅ Processing 1svh ...
1svh: 989 pairs found
✅ Processing 5l2t ...
5l2t: 715 pairs found
✅ Processing 4tv3 ...
4tv3: 609 pairs found
✅ Processing 5m55 ...
5m55: 493 pairs found
✅ Processing 4rx7 ...
4rx7: 684 pairs found
✅ Processing 4qms ...
4qms: 680 pairs found
✅ Processing 2wxl ...
2wxl: 699 pairs found
✅ Processing 3uyt ...
3uyt: 634 pairs found
✅ Processing 4jbo ...
4jbo: 998 pairs found
✅ Processing 4qp7 ...
4qp7: 368 pairs found
✅ Processing 3el7 ...
3el7: 927 pairs found
✅ Processing 5kkr ...
5kkr: 717 pairs found
✅ Processing 1unl ...
1unl: 610 pairs found
✅ Processing 2brg ...
2brg: 572 pairs found
✅ Processing 4jaj ...
4jaj: 379 pairs found
✅ Processing 4xg3 ...
4xg3: 678 

In [81]:
import os
print(os.getcwd())


/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial


Generating ECIF descriptors for external data

In [83]:
import os
import pandas as pd

protein_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/protein_split_pdb"
ligand_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/ligands_split_sdf_rdkit"

ligand_files = [f for f in os.listdir(ligand_dir) if f.endswith(".sdf")]
results = []

for ligand_file in ligand_files:
    prefix = ligand_file.split("_")[0]  # e.g., "1a1b"
    protein_file = os.path.join(protein_dir, f"{prefix}_protein_receptor.pdb")
    ligand_path = os.path.join(ligand_dir, ligand_file)

    if not os.path.exists(protein_file):
        print(f"⚠️ Protein file not found for {prefix}, skipping...")
        continue

    print(f"✅ Processing {prefix} ...")

    # === TRY-EXCEPT to skip any problematic complex ===
    try:
        ecif_vector = GetECIF(protein_file, ligand_path)
        if ecif_vector is None or len(ecif_vector) == 0:
            print(f"⚠️ Skipping {prefix} — no ECIF features generated.")
            continue
        # Add Complex ID as first column
        ecif_vector = [prefix] + ecif_vector
        results.append(ecif_vector)

    except Exception as e:
        print(f"❌ Skipping {prefix} due to error: {e}")
        continue

# Convert to DataFrame
if results:
    # First column = Complex_ID, rest = ECIF headers
    headers = ["Complex_ID"] + PossibleECIF
    df_ecif = pd.DataFrame(results, columns=headers)
    df_ecif.to_csv("ECIF_descriptors_test.csv", index=False)
    print(f"\n✅ ECIF descriptors saved to: ECIF_descriptors_train.csv")
else:
    print("❌ No ECIF features generated. Check your input files.")


✅ Processing 4CWF ...
4CWF: 418 pairs found
✅ Processing 6ZRC ...
6ZRC: 1451 pairs found
✅ Processing 1L5Q ...
1L5Q: 493 pairs found
✅ Processing 5RK9 ...
5RK9: 335 pairs found
✅ Processing 8CPH ...
8CPH: 500 pairs found
✅ Processing 4MGI ...
4MGI: 581 pairs found
✅ Processing 7ER8 ...
7ER8: 1191 pairs found
✅ Processing 3FUR ...
3FUR: 753 pairs found
✅ Processing 8EYO ...
8EYO: 1026 pairs found
✅ Processing 5LO5 ...
5LO5: 829 pairs found
✅ Processing 6NWK ...
6NWK: 749 pairs found
✅ Processing 6B1U ...
6B1U: 798 pairs found
✅ Processing 3FXW ...
3FXW: 633 pairs found
✅ Processing 3NOX ...
3NOX: 592 pairs found
✅ Processing 4O76 ...
4O76: 641 pairs found
✅ Processing 7B31 ...
7B31: 896 pairs found
✅ Processing 5IAW ...
5IAW: 512 pairs found
✅ Processing 8U4W ...
8U4W: 975 pairs found
✅ Processing 3BGP ...
3BGP: 529 pairs found
✅ Processing 5TM6 ...
5TM6: 634 pairs found
✅ Processing 5P9J ...
5P9J: 784 pairs found
✅ Processing 4MGA ...
4MGA: 332 pairs found
✅ Processing 6JOJ ...
6JOJ: 7

Joining Target label pIC to the training descriptors

In [86]:
import pandas as pd

# Load ECIF descriptors
df_ecif = pd.read_csv("ECIF_descriptors_train.csv")  # Contains Complex_ID + ECIF descriptors

# Load activity data
df_activity = pd.read_csv("general_activity_2483.csv")  # Contains PDB_ID and pIC

# Map pIC to ECIF dataframe based on Complex_ID
df_ecif["pIC"] = df_ecif["Complex_ID"].map(df_activity.set_index("PDB_ID")["pIC"])

# Save the merged file
df_ecif.to_csv("ECIF_with_activity_train.csv", index=False)
print("✅ ECIF descriptors with pIC added as last column saved to 'ECIF_with_activity_train.csv'")


✅ ECIF descriptors with pIC added as last column saved to 'ECIF_with_activity_train.csv'


In [88]:
df_ecif.head()

,Complex_ID,C;4;1;3;0;0-Br;1;1;0;0;0,C;4;1;3;0;0-C;3;3;0;1;1,C;4;1;3;0;0-C;4;1;1;0;0,C;4;1;3;0;0-C;4;1;2;0;0,C;4;1;3;0;0-C;4;1;3;0;0,C;4;1;3;0;0-C;4;2;0;0;0,C;4;1;3;0;0-C;4;2;1;0;0,C;4;1;3;0;0-C;4;2;1;0;1,C;4;1;3;0;0-C;4;2;1;1;1,...,S;2;2;0;0;0-S;2;2;0;0;0,S;2;2;0;0;0-S;2;2;0;0;1,S;2;2;0;0;0-S;2;2;0;1;1,S;2;2;0;0;0-S;3;3;0;0;0,S;2;2;0;0;0-S;3;3;0;0;1,S;2;2;0;0;0-S;4;3;0;0;0,S;2;2;0;0;0-S;6;4;0;0;0,S;2;2;0;0;0-S;6;4;0;0;1,S;2;2;0;0;0-S;7;4;0;0;0,pIC
0,3r02,8,0,0,0,0,0,0,27,0,...,0,0,0,0,0,0,0,0,0,9.00
1,6c2t,0,0,0,0,0,0,0,47,0,...,0,0,0,0,0,0,0,0,0,9.05
2,5t31,0,0,0,0,8,0,0,20,0,...,0,0,0,0,0,0,0,0,0,6.96
3,4eop,0,0,0,0,0,0,8,49,0,...,0,0,0,0,0,0,0,0,0,6.05
4,2ghg,0,0,0,0,0,0,6,38,0,...,0,0,0,0,0,0,0,0,0,8.82


Generating RDKit descriptors for training dataset

In [16]:
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
import pandas as pd
import os
from tqdm import tqdm

# Define directories
ligand_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/ligand_train_rdkit"
save_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_train.csv"

# List all ligand files
ligand_files = [f for f in os.listdir(ligand_dir) if f.endswith('.sdf') or f.endswith('.mol')]

# Get list of RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Storage list
data = []

# Loop through ligands
for file in tqdm(ligand_files, desc="Calculating RDKit descriptors"):
    file_path = os.path.join(ligand_dir, file)
    mol = Chem.MolFromMolFile(file_path)
    
    if mol is None:
        print(f"⚠️ Skipping {file} (cannot read molecule)")
        continue
    
    # Calculate all RDKit descriptors
    desc_values = calculator.CalcDescriptors(mol)
    
    # Append results
    complex_id = os.path.splitext(file)[0]  # use filename as Complex_ID
    data.append([complex_id] + list(desc_values))

# Create DataFrame
desc_df = pd.DataFrame(data, columns=["Complex_ID"] + descriptor_names)

# Save to CSV
desc_df.to_csv(save_path, index=False)
print(f"RDKit descriptors saved successfully at:\n{save_path}")


Calculating RDKit descriptors:   3%|▎         | 70/2401 [00:00<00:26, 86.56it/s]

⚠️ Skipping 2wd1_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors:   4%|▎        | 100/2401 [00:01<00:25, 90.17it/s]

⚠️ Skipping 2c69_ligand_rdkit.sdf (cannot read molecule)
⚠️ Skipping 2w1e_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors:   5%|▍        | 130/2401 [00:01<00:25, 90.70it/s]

...

⚠️ Skipping 3r9n_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors:  99%|██████▉| 2386/2401 [00:24<00:00, 100.94it/s]

⚠️ Skipping 5wjj_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors: 100%|████████| 2401/2401 [00:24<00:00, 97.74it/s]


⚠️ Skipping 3hv5_ligand_rdkit.sdf (cannot read molecule)
✅ RDKit descriptors saved successfully at:
/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_train.csv


Generating RDKit descriptors for external dataset

In [17]:
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
import pandas as pd
import os
from tqdm import tqdm

# Define directories
ligand_dir = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/ligands_split_sdf_rdkit"
save_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_test.csv"

# List all ligand files
ligand_files = [f for f in os.listdir(ligand_dir) if f.endswith('.sdf') or f.endswith('.mol')]

# Get list of RDKit descriptors
descriptor_names = [desc[0] for desc in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Storage list
data = []

# Loop through ligands
for file in tqdm(ligand_files, desc="Calculating RDKit descriptors"):
    file_path = os.path.join(ligand_dir, file)
    mol = Chem.MolFromMolFile(file_path)
    
    if mol is None:
        print(f"⚠️ Skipping {file} (cannot read molecule)")
        continue
    
    # Calculate all RDKit descriptors
    desc_values = calculator.CalcDescriptors(mol)
    
    # Append results
    complex_id = os.path.splitext(file)[0]  # use filename as Complex_ID
    data.append([complex_id] + list(desc_values))

# Create DataFrame
desc_df = pd.DataFrame(data, columns=["Complex_ID"] + descriptor_names)

# Save to CSV
desc_df.to_csv(save_path, index=False)
print(f"✅ RDKit descriptors saved successfully at:\n{save_path}")


Calculating RDKit descriptors:   1%|          | 37/6340 [00:00<01:48, 58.35it/s]

⚠️ Skipping 6LX4_protein_ligand_rdkit.sdf (cannot read molecule)
⚠️ Skipping 8A4N_protein_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors:   1%|▏         | 85/6340 [00:01<01:18, 79.36it/s]

⚠️ Skipping 3V2Y_protein_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors:   2%|▏        | 114/6340 [00:01<01:13, 84.24it/s]

...

Calculating RDKit descriptors:  99%|██████▉| 6297/6340 [01:02<00:00, 120.82it/s]

⚠️ Skipping 6UVC_protein_ligand_rdkit.sdf (cannot read molecule)
⚠️ Skipping 2SHP_protein_ligand_rdkit.sdf (cannot read molecule)


Calculating RDKit descriptors: 100%|███████| 6340/6340 [01:02<00:00, 101.11it/s]

⚠️ Skipping 4UDD_protein_ligand_rdkit.sdf (cannot read molecule)
⚠️ Skipping 2R3Q_protein_ligand_rdkit.sdf (cannot read molecule)
⚠️ Skipping 4R6S_protein_ligand_rdkit.sdf (cannot read molecule)


✅ RDKit descriptors saved successfully at:
/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_test.csv


In [9]:
import pandas as pd
df_rdkit_test = pd.read_csv("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_test.csv")
df_rdkit_test.head()

,Complex_ID,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,4CWF_protein_ligand_rdkit,5.675801,5.675801,0.292135,0.292135,0.724529,11.352941,227.271,214.167,227.117095,...,0,0,0,0,0,0,0,0,0,0
1,6ZRC_protein_ligand_rdkit,15.210403,15.210403,0.009959,-1.702824,0.017090,24.666667,1769.205,1637.157,1767.959951,...,2,0,0,0,0,0,0,0,2,0
2,1L5Q_protein_ligand_rdkit,10.731864,10.731864,0.445065,-1.465046,0.337239,41.266667,221.209,206.089,221.089937,...,0,0,0,0,0,0,0,0,0,0
3,5RK9_protein_ligand_rdkit,11.071481,11.071481,0.033565,-0.033565,0.600005,9.400000,139.158,130.086,139.074562,...,0,0,0,0,0,0,0,0,0,0
4,8CPH_protein_ligand_rdkit,10.590634,10.590634,0.107882,-0.923506,0.496581,10.428571,323.805,309.693,323.049525,...,1,0,0,0,0,0,0,0,0,0


In [11]:
print(list(df_rdkit_test.columns))

['Complex_ID', 'MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'AvgIpc', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA8', 'SMR_VSA9', 'SlogP_VSA1', 'Sl

Finding out overlapping complexes between training and external datasets

In [19]:
import pandas as pd

# File paths
train_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_train.csv"
test_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test.csv"

# Load both datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Extract Complex_ID columns
train_ids = set(train_df["Complex_ID"].astype(str).str.lower())
test_ids = set(test_df["Complex_ID"].astype(str).str.lower())

# Find common IDs
common_ids = train_ids.intersection(test_ids)

# Print results
print(f"✅ Number of common Complex_IDs: {len(common_ids)}")
if common_ids:
    print("Common IDs:")
    print(common_ids)
else:
    print("🎉 No common Complex_IDs found between train and test!")


✅ Number of common Complex_IDs: 770
Common IDs:
{'4lm5', '3vby', '2xf0', '5c26', '3fyj', '4jlm', '2wmw', '5d7a', '4kcg', '4cmu', '4ot6', '4bcf', '2i6a', '3fv8', '3vbt', '2vto', '3byu', '5idp', '4y73', '4zts', '4q9s', '4y46', '4hvb', '2wu6', '4jx7', '4yne', '3e7o', '6ayd', '3kc3', '4d2t', '5t18', '5jrs', '4qmm', '4hnf', '4fny', '4qmv', '2p3g', '6fil', '5kbq', '5t1s', '4otg', '6e2n', '5vcx', '3kvw', '2ym7', '4l02', '1gi9', '5bnj', '3tku', '5kpm', '4joa', '3zim', '6fyp', '5opu', '1pye', '4bkj', '4alv', '4c4e', '4oth', '4ppb', '3cy2', '4bci', '5jq8', '3qd3', '3r01', '1h1p', '4wsy', '5ftq', '3gp0', '6dkg', '4y5h', '6ft8', '6mx8', '1pxk', '5v24', '3rtp', '5csw', '5gjd', '4v04', '4hys', '5a4q', '3vbq', '1o4n', '4at3', '4mq2', '2wih', '5l6o', '2w1h', '4cfe', '3dpd', '3v3v', '4rpv', '4mp2', '1uwh', '3nw6', '3prz', '4hcu', '5lxc', '3tti', '4aw5', '4i0t', '3zw3', '6db3', '4y83', '5jga', '4q1b', '3fi2', '5hfu', '4hgl', '4zjj', '5tc0', '4yti', '1unl', '4erw', '3e3b', '3p9j', '3g90', '6nfz', '4d2s',

Removing overlapping complexes from external dataset ECIF descriptors

In [22]:
import pandas as pd

# File paths
train_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_train.csv"
test_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test.csv"
output_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv"

# Load data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Convert Complex_ID to lowercase for matching
train_ids = set(train_df["Complex_ID"].astype(str).str.lower())
test_df["Complex_ID_lower"] = test_df["Complex_ID"].astype(str).str.lower()

# Filter out overlapping Complex_IDs
test_df_clean = test_df[~test_df["Complex_ID_lower"].isin(train_ids)].copy()

# Drop helper column
test_df_clean.drop(columns=["Complex_ID_lower"], inplace=True)

# Save the cleaned test file
test_df_clean.to_csv(output_path, index=False)

print(f" Cleaned test file saved at: {output_path}")
print(f" Removed {len(test_df) - len(test_df_clean)} overlapping Complex_IDs")
print(f"Final test set size: {len(test_df_clean)}")


✅ Cleaned test file saved at: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv
🧹 Removed 770 overlapping Complex_IDs
✅ Final test set size: 5265


Merging ECIF and RDKit descriptors of training dataset

In [1]:
import pandas as pd

# File paths
ecif_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_with_activity_train.csv"
rdkit_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_train.csv"

# Load both dataframes
df_ecif = pd.read_csv(ecif_path)
df_rdkit = pd.read_csv(rdkit_path)

# Clean the RDKit Complex_ID column by removing '_ligand_rdkit'
df_rdkit["Complex_ID"] = df_rdkit["Complex_ID"].str.replace("_ligand_rdkit", "", regex=False)

# Merge based on Complex_ID
merged_df = pd.merge(df_ecif, df_rdkit, on="Complex_ID", how="inner")

# Save merged dataframe
merged_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_train_full.csv"
merged_df.to_csv(merged_path, index=False)

print(f"✅ Merged file saved successfully: {merged_path}")
print("Merged dataframe shape:", merged_df.shape)
print("Example rows:")
print(merged_df.head())


✅ Merged file saved successfully: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_train_full.csv
Merged dataframe shape: (2337, 1759)
Example rows:
  Complex_ID  C;4;1;3;0;0-Br;1;1;0;0;0  C;4;1;3;0;0-C;3;3;0;1;1  \
0       3r02                         8                        0   
1       6c2t                         0                        0   
2       5t31                         0                        0   
3       4eop                         0                        0   
4       2ghg                         0                        0   

   C;4;1;3;0;0-C;4;1;1;0;0  C;4;1;3;0;0-C;4;1;2;0;0  C;4;1;3;0;0-C;4;1;3;0;0  \
0                        0                        0                        0   
1                        0                        0                        0   
2                        0                        0                        8   
3                        0                        0                        0   
4                  

Merging ECIF and RDKit descriptors of external dataset

In [27]:
import pandas as pd

# File paths
ecif_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test.csv"
rdkit_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/rdkit_descriptors_test.csv"

# Load both dataframes
df_ecif = pd.read_csv(ecif_path)
df_rdkit = pd.read_csv(rdkit_path)

# Clean the RDKit Complex_ID column by removing '_ligand_rdkit'
df_rdkit["Complex_ID"] = df_rdkit["Complex_ID"].str.replace("_protein_ligand_rdkit", "", regex=False)

# Merge based on Complex_ID
merged_df = pd.merge(df_ecif, df_rdkit, on="Complex_ID", how="inner")

# Save merged dataframe
merged_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test.csv"
merged_df.to_csv(merged_path, index=False)

print(f"✅ Merged file saved successfully: {merged_path}")
print("Merged dataframe shape:", merged_df.shape)
print("Example rows:")
print(merged_df.head())


✅ Merged file saved successfully: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test.csv
Merged dataframe shape: (6035, 1758)
Example rows:
  Complex_ID  C;4;1;3;0;0-Br;1;1;0;0;0  C;4;1;3;0;0-C;3;3;0;1;1  \
0       4CWF                         0                        0   
1       6ZRC                         0                        0   
2       1L5Q                         0                        0   
3       5RK9                         0                        0   
4       8CPH                         0                        0   

   C;4;1;3;0;0-C;4;1;1;0;0  C;4;1;3;0;0-C;4;1;2;0;0  C;4;1;3;0;0-C;4;1;3;0;0  \
0                        0                        0                        3   
1                        0                        0                        9   
2                        0                        0                        3   
3                        0                        0                        8   
4                        

Removing overlapping complexes from external dataset ECIF and RDKit merged descriptors

In [3]:
import pandas as pd

# File paths
train_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/merged_clean.csv"
test_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test.csv"
output_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test_nonoverlapped.csv"

# Load data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Convert Complex_ID to lowercase for matching
train_ids = set(train_df["Complex_ID"].astype(str).str.lower())
test_df["Complex_ID_lower"] = test_df["Complex_ID"].astype(str).str.lower()

# Filter out overlapping Complex_IDs
test_df_clean = test_df[~test_df["Complex_ID_lower"].isin(train_ids)].copy()

# Drop helper column
test_df_clean.drop(columns=["Complex_ID_lower"], inplace=True)

# Save the cleaned test file
test_df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned test file saved at: {output_path}")
print(f"🧹 Removed {len(test_df) - len(test_df_clean)} overlapping Complex_IDs")
print(f"✅ Final test set size: {len(test_df_clean)}")


✅ Cleaned test file saved at: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test_nonoverlapped.csv
🧹 Removed 770 overlapping Complex_IDs
✅ Final test set size: 5265


In [5]:
import pandas as pd
test = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test_nonoverlapped.csv"
test_data = pd.read_csv(test)
test_data.shape

(5265, 1758)